In [1]:
import os, json, random, datetime
from typing import List, Dict, Any
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Seed:", SEED, "| CUDA available:", torch.cuda.is_available())

CAI_DIR = "sft_data_icai"
os.makedirs(CAI_DIR, exist_ok=True)

SL_PROMPTS_PATH   = os.path.join(CAI_DIR, "alert_sl_prompts.jsonl")
PREF_PROMPTS_PATH = os.path.join(CAI_DIR, "alert_pref_prompts.jsonl")
TEST_PROMPTS_PATH = os.path.join(CAI_DIR, "alert_test_prompts.jsonl")

SFT_CRITIQUE_AND_REVISION = os.path.join(CAI_DIR, "sft_revision_and_critique.jsonl")
SFT_CRITIQUE_AND_REVISION_META_PATH    = os.path.join(CAI_DIR, "sft_revision_and_critique_meta.json")

PRINCIPLES_PATH = ("icai_principles.json")

HF_DATASET = "Babelscape/ALERT"
TOTAL_PROMPTS = 10000
SPLIT_FRACTION   = 0.2 

MISTRAL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"
DEVICE_MAP = "auto"
print("CAI_DIR:", CAI_DIR)
print("Model:", MISTRAL_NAME)
print("Dataset:", HF_DATASET, "| TOTAL_PROMPTS:", TOTAL_PROMPTS, "| SPLIT_FRACTION:", SPLIT_FRACTION)

Seed: 42 | CUDA available: True
CAI_DIR: sft_data_icai
Model: mistralai/Mistral-7B-Instruct-v0.1
Dataset: Babelscape/ALERT | TOTAL_PROMPTS: 2000 | SL_FRACTION: 0.5


In [3]:
def load_principles(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    summ_principles = []
    for d in data:
        summ_principles.append(d["principle"])
    return summ_principles

principles_list: List[str] = load_principles(PRINCIPLES_PATH)
print(f"Loaded {len(principles_list)} principles.")


Loaded 15 principles.


In [4]:
print("Loading HF dataset:", HF_DATASET)
raw = load_dataset(HF_DATASET, name="alert")  
split_name = "train"
ds = raw[split_name].shuffle(seed=SEED) 

def extract_prompt(item):
    if isinstance(item, dict):
        return str(item["prompt"])
    return json.dumps(item, ensure_ascii=False)

Loading HF dataset: Babelscape/ALERT


README.md: 0.00B [00:00, ?B/s]

alert.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/14763 [00:00<?, ? examples/s]

In [5]:
TOTAL = min(TOTAL_PROMPTS, len(ds))
if TOTAL < TOTAL_PROMPTS:
    print(f"Warning: dataset size {len(ds)} < requested TOTAL_PROMPTS {TOTAL_PROMPTS}. Using {TOTAL} examples.")

ds_small = ds.select(range(TOTAL))

sl_boundary = int(TOTAL * SPLIT_FRACTION)
test_boundary = int(TOTAL * SPLIT_FRACTION) + sl_boundary
sl_split = ds_small.select(range(sl_count))
test_split = ds_small.select(range(sl_count, test_boundary))
pref_split = ds_small.select(range(test_boundary, TOTAL))

print(f"Subset sizes -> SL: {len(sl_split)}, PREF: {len(pref_split)}, TEST: {len(test_split)}, (TOTAL={TOTAL})")

Subset sizes -> SL: 1000, PREF: 1000 (TOTAL=2000)


In [6]:
def save_jsonl(dataset, path: str):
    with open(path, "w", encoding="utf-8") as f:
        for item in dataset:
            if hasattr(item, "items"):
                item = dict(item)
            rec = {
                "id": item.get("id") if isinstance(item, dict) else None,
                "category": item.get("category") if isinstance(item, dict) else None,
                "prompt": extract_prompt(item),
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl(sl_split, SL_PROMPTS_PATH)
save_jsonl(pref_split, PREF_PROMPTS_PATH)
save_jsonl(test_split, TEST_PROMPTS_PATH)
print("Saved SFT prompts ->", SL_PROMPTS_PATH)
print("Saved PREF prompts ->", PREF_PROMPTS_PATH)
print("Saved TEST prompts ->", TEST_PROMPTS_PATH)

Saved SL prompts -> sft_data_icai/alert_sl_prompts.jsonl
Saved PREF prompts -> sft_data_icai/alert_pref_prompts.jsonl


In [7]:
class HFProvider:
    def __init__(self, model_name: str, device_map="auto"):
        tok = AutoTokenizer.from_pretrained(model_name)
         
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token

        self.tok = tok

        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map=device_map,
        )
        self.model.config.pad_token_id = self.tok.pad_token_id
        self.model.generation_config.pad_token_id = self.tok.pad_token_id
        self.model.generation_config.eos_token_id = self.tok.eos_token_id

    def generate(self, user_prompt: str, max_new_tokens=1024, temperature=0.7, top_p=0.95) -> str:
        input_ids = self.tok.apply_chat_template(
            [{"role": "user", "content": user_prompt}],
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(self.model.device)
        attention_mask = torch.ones_like(input_ids)

        with torch.no_grad():
            out = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,                
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=self.tok.pad_token_id,            
                eos_token_id=self.tok.eos_token_id,
            )

        # Decode only the generated continuation
        gen = out[0][input_ids.shape[-1]:]
        return self.tok.decode(gen, skip_special_tokens=True).strip()

model = HFProvider(MISTRAL_NAME, device_map=DEVICE_MAP)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [8]:
INITIAL_TEMPLATE = (
    "You are a helpful assistant. Provide your best **safe** answer.\n"
    "User request:\n{prompt}\n"
    "Assistant answer:"
)

CRITIQUE_TEMPLATE = (
    "You are a critique assistant applying the following constitutional principle to evaluate the prior answer.\n\n"
    "Principle:\n{principle}\n\n"
    "Task: Identify issues with the answer relative to this principle. Be concise but specific.\n"
    "Return JSON with keys: rationale, required_changes (list of short bullets).\n"
    "Answer to critique:\n{answer}\n"
    "JSON:"
)

REVISION_TEMPLATE = (
    "You are a reviser. Improve the assistant's answer using ONLY the specific critique below.\n\n"
    "Critique JSON:\n{critique_json}\n\n"
    "User request:\n{prompt}\n\n"
    "Original answer:\n{answer}\n\n"
    "Write a revised answer that addresses the critique while refusing unsafe requests. Keep it short and clear. Do not reference the initial answer directly.\n"
    "Revised answer:"
)

In [9]:
def load_jsonl(path: str):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            items.append(json.loads(line))
    return items

sl_prompts = load_jsonl(SL_PROMPTS_PATH)
print("SL prompts loaded:", len(sl_prompts))

SL prompts loaded: 1000


In [10]:
sl_records = []
N = len(sl_prompts)
rng = np.random.default_rng(SEED)

for i, ex in enumerate(sl_prompts, 1):
    prompt = ex["prompt"]
    ex_id = ex.get("id")
    category = ex.get("category")

    init_answer = model.generate(
        INITIAL_TEMPLATE.format(prompt=prompt),
        max_new_tokens=512, temperature=0.7, top_p=0.95
    )

    p_idx = int(rng.integers(low=0, high=len(principles_list)))
    p_text = principles_list[p_idx]

    critique_prompt = CRITIQUE_TEMPLATE.format(principle=p_text, answer=init_answer)
    critique_json = model.generate(
        critique_prompt, max_new_tokens=512, temperature=0.2, top_p=0.9
    )

    rev_prompt = REVISION_TEMPLATE.format(critique_json=critique_json, prompt=prompt, answer=init_answer)
    revised = model.generate(
        rev_prompt, max_new_tokens=512, temperature=0.7, top_p=0.95
    )

    sl_records.append({
        "id": ex_id,
        "category": category,
        "prompt": prompt,
        "initial_answer": init_answer,
        "critiques": [{
            "principle_index": p_idx,
            "principle_text": p_text,
            "critique_json": critique_json
        }],
        "revisions": [{
            "principle_index": p_idx,
            "revised": revised
        }]
    })

    if i % 10 == 0 or i == N:
        print(f"[{i}/{N}]")


with open(SFT_CRITIQUE_AND_REVISION, "w", encoding="utf-8") as f:
    for r in sl_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

meta = {
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "model": MISTRAL_NAME,
    "num_examples": len(sl_records),
    "dataset": HF_DATASET,
    "total_prompts": 2000,
    "SPLIT_FRACTION": 0.5
}
with open(SFT_CRITIQUE_AND_REVISION_META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("Wrote:", SFT_CRITIQUE_AND_REVISION, " | records:", len(sl_records))
print("Meta:", SFT_CRITIQUE_AND_REVISION_META_PATH)

[10/1000]
[20/1000]
[30/1000]
[40/1000]
[50/1000]
[60/1000]
[70/1000]
[80/1000]
[90/1000]
[100/1000]
[110/1000]
[120/1000]
[130/1000]
[140/1000]
[150/1000]
[160/1000]
[170/1000]
[180/1000]
[190/1000]
[200/1000]
[210/1000]
[220/1000]
[230/1000]
[240/1000]
[250/1000]
[260/1000]
[270/1000]
[280/1000]
[290/1000]
[300/1000]
[310/1000]
[320/1000]
[330/1000]
[340/1000]
[350/1000]
[360/1000]
[370/1000]
[380/1000]
[390/1000]
[400/1000]
[410/1000]
[420/1000]
[430/1000]
[440/1000]
[450/1000]
[460/1000]
[470/1000]
[480/1000]
[490/1000]
[500/1000]
[510/1000]
[520/1000]
[530/1000]
[540/1000]
[550/1000]
[560/1000]
[570/1000]
[580/1000]
[590/1000]
[600/1000]
[610/1000]
[620/1000]
[630/1000]
[640/1000]
[650/1000]
[660/1000]
[670/1000]
[680/1000]
[690/1000]
[700/1000]
[710/1000]
[720/1000]
[730/1000]
[740/1000]
[750/1000]
[760/1000]
[770/1000]
[780/1000]
[790/1000]
[800/1000]
[810/1000]
[820/1000]
[830/1000]
[840/1000]
[850/1000]
[860/1000]
[870/1000]
[880/1000]
[890/1000]
[900/1000]
[910/1000]
[920/100

/tmp/ipykernel_5234/1021108882.py:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
